In [3]:
# Cell 1 — Imports and path setup
import torch
import numpy as np
import pandas as pd
from pathlib import Path

ROOT      = Path("interpretability/outputs")
SAE_BASE  = ROOT / "sae"
STEER_BASE = ROOT / "steering"
PROP_BASE  = STEER_BASE / "property_sweep"
DOSE_BASE  = STEER_BASE / "dose_response"

CONFIGS = [
    (2048, 16), (2048, 32), (2048, 64),
    (4096, 16), (4096, 32), (4096, 64),
    (8192, 16), (8192, 32), (8192, 64),
]
TOP_NS  = [10, 50, 100, 200, 400]
SCALES  = [0.5, 1.0, 2.0, 3.0, 5.0]

WINNERS = [
    ("nf2048_k64", "8×",  2048, 64, 400),
    ("nf4096_k64", "16×", 4096, 64,  50),
    ("nf8192_k32", "32×", 8192, 32, 100),
]

def kl_divergence(base_dist, steer_dist):
    all_elems = set(base_dist.keys()) | set(steer_dist.keys())
    b_total = max(sum(base_dist.values()), 1)
    s_total = max(sum(steer_dist.values()), 1)
    eps = 1e-10
    kl = 0.0
    for e in all_elems:
        p = steer_dist.get(e, 0) / s_total + eps
        q = base_dist.get(e, 0) / b_total + eps
        kl += p * np.log(p / q)
    return kl

print("Paths:")
print(f"  SAE:        {SAE_BASE}")
print(f"  Prop sweep: {PROP_BASE}")
print(f"  Dose resp:  {DOSE_BASE}")

Paths:
  SAE:        interpretability/outputs/sae
  Prop sweep: interpretability/outputs/steering/property_sweep
  Dose resp:  interpretability/outputs/steering/dose_response


In [4]:
# Cell 2 — SAE ablation stats (R², dead, rare) for all 9 configs
sae_stats = {}

print(f"{'n':>6} {'k':>4} {'R2_norm':>8} {'R2_raw':>8} {'Dead':>6} {'Rare':>6}")
print("-" * 45)

for n, k in CONFIGS:
    path = SAE_BASE / f"nf{n}_k{k}" / "analysis_z_con.pt"
    d    = torch.load(path, map_location="cpu", weights_only=False)
    r2_norm = d["history"]["var_explained"][-1]
    r2_raw  = d["history"]["var_explained_raw"][-1]
    dead    = int(d["n_dead"])
    rare    = int(d["n_rare"])
    sae_stats[(n, k)] = dict(r2_norm=r2_norm, r2_raw=r2_raw, dead=dead, rare=rare)
    print(f"{n:>6} {k:>4} {r2_norm:>8.4f} {r2_raw:>8.4f} {dead:>6} {rare:>6}")

     n    k  R2_norm   R2_raw   Dead   Rare
---------------------------------------------
  2048   16   0.8185   0.8364     55   1837
  2048   32   0.8523   0.8664      0   1728
  2048   64   0.8507   0.8606      0    581
  4096   16   0.8193   0.8358    279   3853
  4096   32   0.8449   0.8581      1   3816
  4096   64   0.8398   0.8495      0   2598
  8192   16   0.8128   0.8282   1154   7915
  8192   32   0.8324   0.8450     32   7952
  8192   64   0.8123   0.8208      0   7309


In [5]:
# Cell 3 — Top-N sweep results for all 9 configs × 5 top_n values
sweep_results = {}   # keyed (n, k, top_n)

print(f"{'n':>6} {'k':>4} {'top_n':>6} {'R2_norm':>8} {'KL':>7} {'SMACT_b':>8} {'SMACT_s':>8} {'delta':>7}")
print("-" * 65)

for n, k in CONFIGS:
    r2 = sae_stats[(n, k)]["r2_norm"]
    for top_n in TOP_NS:
        pts = list((PROP_BASE / f"nf{n}_k{k}" / f"top_n{top_n}").glob("steer_amplify_*.pt"))
        if not pts:
            print(f"{n:>6} {k:>4} {top_n:>6}  MISSING")
            continue
        d       = torch.load(pts[0], map_location="cpu", weights_only=False)
        kl      = d["steered"]["comparison"]["kl_divergence"]
        smact_b = d["baseline"]["validity"]["smact_rate"]
        smact_s = d["steered"]["validity"]["smact_rate"]
        delta   = (smact_s - smact_b) * 100
        top_el  = [e for e, _ in d["steered"]["comparison"]["top_enriched"][:3]]
        sweep_results[(n, k, top_n)] = dict(
            r2=r2, kl=kl, smact_b=smact_b, smact_s=smact_s,
            delta=delta, top_elems=top_el
        )
        print(f"{n:>6} {k:>4} {top_n:>6} {r2:>8.4f} {kl:>7.4f} "
              f"{smact_b*100:>7.1f}% {smact_s*100:>7.1f}% {delta:>+6.1f}%")
    print()

     n    k  top_n  R2_norm      KL  SMACT_b  SMACT_s   delta
-----------------------------------------------------------------
  2048   16     10   0.8185  0.5057    85.0%    92.5%   +7.5%
  2048   16     50   0.8185  0.9695    87.5%    87.5%   +0.0%
  2048   16    100   0.8185  1.7996    95.0%    85.0%  -10.0%
  2048   16    200   0.8185  0.8824    87.5%    82.5%   -5.0%
  2048   16    400   0.8185  1.6134    90.0%    87.5%   -2.5%

  2048   32     10   0.8523  1.1767    90.0%    87.5%   -2.5%
  2048   32     50   0.8523  1.7559    90.0%    90.0%   +0.0%
  2048   32    100   0.8523  0.9757    87.5%    80.0%   -7.5%
  2048   32    200   0.8523  1.0207    87.5%    82.5%   -5.0%
  2048   32    400   0.8523  1.6280    90.0%    87.5%   -2.5%

  2048   64     10   0.8507  1.0014    85.0%    90.0%   +5.0%
  2048   64     50   0.8507  1.7066    90.0%    92.5%   +2.5%
  2048   64    100   0.8507  2.1215    92.5%    82.5%  -10.0%
  2048   64    200   0.8507  1.4235    92.5%    92.5%   +0.0%
  

In [6]:
# Cell 4 — N* selection: best valid candidate per config, group winners highlighted
print("N* selection (SMACT Δ ≥ 0, then max KL, R²_norm as tiebreaker)\n")

group_winners = {}  # keyed by n

for n, k in CONFIGS:
    valid = {
        top_n: sweep_results[(n, k, top_n)]
        for top_n in TOP_NS
        if (n, k, top_n) in sweep_results
        and sweep_results[(n, k, top_n)]["delta"] >= 0
    }
    if not valid:
        best_top_n, best = None, None
    else:
        best_top_n = max(valid, key=lambda t: valid[t]["kl"])
        best = valid[best_top_n]

    marker = ""
    if best and (n not in group_winners or
                 best["kl"] > sweep_results[(n,
                     group_winners[n][0],
                     group_winners[n][1])]["kl"]):
        group_winners[n] = (k, best_top_n)
        marker = "  ← group winner"

    if best:
        print(f"n={n:>5} k={k:>3}  N*={best_top_n:>4}  "
              f"KL={best['kl']:.4f}  SMACT Δ={best['delta']:+.1f}%  "
              f"R²={best['r2']:.4f}{marker}")
    else:
        print(f"n={n:>5} k={k:>3}  no valid candidate")

print(f"\nFinal group winners:")
for n, (k, top_n) in sorted(group_winners.items()):
    r = sweep_results[(n, k, top_n)]
    print(f"  {n}/k={k}/top_n={top_n}  KL={r['kl']:.4f}  "
          f"SMACT Δ={r['delta']:+.1f}%  R²={r['r2']:.4f}")

N* selection (SMACT Δ ≥ 0, then max KL, R²_norm as tiebreaker)

n= 2048 k= 16  N*=  50  KL=0.9695  SMACT Δ=+0.0%  R²=0.8185  ← group winner
n= 2048 k= 32  N*=  50  KL=1.7559  SMACT Δ=+0.0%  R²=0.8523  ← group winner
n= 2048 k= 64  N*= 400  KL=2.0736  SMACT Δ=+10.0%  R²=0.8507  ← group winner
n= 4096 k= 16  N*= 400  KL=2.2879  SMACT Δ=+0.0%  R²=0.8193  ← group winner
n= 4096 k= 32  N*= 200  KL=1.6612  SMACT Δ=+7.5%  R²=0.8449
n= 4096 k= 64  N*=  50  KL=2.3996  SMACT Δ=+0.0%  R²=0.8398  ← group winner
n= 8192 k= 16  N*= 400  KL=2.2802  SMACT Δ=+7.5%  R²=0.8128  ← group winner
n= 8192 k= 32  N*= 100  KL=2.4439  SMACT Δ=+7.5%  R²=0.8324  ← group winner
n= 8192 k= 64  N*=  10  KL=0.9771  SMACT Δ=+0.0%  R²=0.8123

Final group winners:
  2048/k=64/top_n=400  KL=2.0736  SMACT Δ=+10.0%  R²=0.8507
  4096/k=64/top_n=50  KL=2.3996  SMACT Δ=+0.0%  R²=0.8398
  8192/k=32/top_n=100  KL=2.4439  SMACT Δ=+7.5%  R²=0.8324


In [7]:
# Cell 5 — Dose-response results for 3 winners
dose_results = {}   # keyed (folder, scale)

for folder, exp, n, k, top_n in WINNERS:
    b = torch.load(DOSE_BASE / folder / "steer_sweep_formation_energy_scale1_0.pt",
                   map_location="cpu", weights_only=False)
    base_dist  = b["results"]["element_distribution"]
    smact_base = b["results"]["validity"]["smact_rate"]

    print(f"\n{'='*65}")
    print(f"{exp} | {folder} | k={k} | N*={top_n}")
    print(f"{'='*65}")
    print(f"{'scale':>6} {'KL':>8} {'SMACT':>8} {'SMACT Δ':>10}  top elements")
    print("-" * 65)

    dose_results[folder] = {}
    for scale in SCALES:
        scale_str = f"{scale:.1f}".replace(".", "_")
        path = DOSE_BASE / folder / f"steer_sweep_formation_energy_scale{scale_str}.pt"
        d          = torch.load(path, map_location="cpu", weights_only=False)
        r          = d["results"]
        steer_dist = r["element_distribution"]
        smact_s    = r["validity"]["smact_rate"]
        delta      = (smact_s - smact_base) * 100
        kl         = kl_divergence(base_dist, steer_dist)
        top_el     = ", ".join(e for e, _ in
                     sorted(steer_dist.items(), key=lambda x: -x[1])[:3])

        dose_results[folder][scale] = dict(
            kl=kl, smact=smact_s, delta=delta, top_elems=top_el
        )
        print(f"{scale:>6.1f} {kl:>8.4f} {smact_s*100:>7.1f}% {delta:>+9.1f}%  {top_el}")


8× | nf2048_k64 | k=64 | N*=400
 scale       KL    SMACT    SMACT Δ  top elements
-----------------------------------------------------------------
   0.5   2.0306    87.5%      +0.0%  Cu, Mg, In
   1.0   0.0000    87.5%      +0.0%  Fe, Ni, H
   2.0   2.1184    90.0%      +2.5%  Fe, Ag, Ni
   3.0   1.5608    87.5%      +0.0%  Ni, Cu, Fe
   5.0   3.0426    85.0%      -2.5%  Ni, Fe, N

16× | nf4096_k64 | k=64 | N*=50
 scale       KL    SMACT    SMACT Δ  top elements
-----------------------------------------------------------------
   0.5   0.6159    90.0%      +2.5%  Fe, Mg, Ni
   1.0   0.0000    87.5%      +0.0%  Fe, Mg, Sn
   2.0   1.1086    90.0%      +2.5%  Cu, Ni, Co
   3.0   1.4964    90.0%      +2.5%  Fe, Ni, Co
   5.0   2.8864    80.0%      -7.5%  Co, Ni, Cu

32× | nf8192_k32 | k=32 | N*=100
 scale       KL    SMACT    SMACT Δ  top elements
-----------------------------------------------------------------
   0.5   0.5257    90.0%      +7.5%  Fe, Mg, Ni
   1.0   0.0000    82.5%  

In [8]:
# Cell 6 — Final winner selection across 3 group winners
# Rule: same as before — SMACT Δ ≥ 0, then max KL, then R²_norm as tiebreaker

print("Final winner selection (scale* = best valid scale per winner)\n")
print(f"{'folder':<15} {'exp':>4} {'scale*':>7} {'KL':>8} {'SMACT Δ':>10} {'R²_norm':>8}")
print("-" * 60)

final_candidates = []

for folder, exp, n, k, top_n in WINNERS:
    r2 = sae_stats[(n, k)]["r2_norm"]

    # best valid scale: SMACT Δ ≥ 0, then max KL
    valid_scales = {
        scale: dose_results[folder][scale]
        for scale in SCALES
        if dose_results[folder][scale]["delta"] >= 0
    }
    if not valid_scales:
        print(f"{folder:<15}  no valid scale")
        continue

    best_scale = max(valid_scales, key=lambda s: valid_scales[s]["kl"])
    best       = valid_scales[best_scale]

    final_candidates.append(dict(
        folder=folder, exp=exp, n=n, k=k, top_n=top_n,
        scale=best_scale, kl=best["kl"], delta=best["delta"], r2=r2
    ))
    print(f"{folder:<15} {exp:>4} {best_scale:>7.1f} {best['kl']:>8.4f} "
          f"{best['delta']:>+9.1f}% {r2:>8.4f}")

# overall winner: max KL among valid candidates, R² as tiebreaker
overall = max(final_candidates, key=lambda x: (x["kl"], x["r2"]))

print(f"\n{'='*60}")
print(f"OVERALL WINNER")
print(f"{'='*60}")
print(f"  Config:  n={overall['n']}, k={overall['k']}")
print(f"  N*:      top_n={overall['top_n']}")
print(f"  Scale*:  {overall['scale']}")
print(f"  KL:      {overall['kl']:.4f}")
print(f"  SMACT Δ: {overall['delta']:+.1f}%")
print(f"  R²_norm: {overall['r2']:.4f}")
print(f"  Folder:  {overall['folder']}")

Final winner selection (scale* = best valid scale per winner)

folder           exp  scale*       KL    SMACT Δ  R²_norm
------------------------------------------------------------
nf2048_k64        8×     2.0   2.1184      +2.5%   0.8507
nf4096_k64       16×     3.0   1.4964      +2.5%   0.8398
nf8192_k32       32×     2.0   1.6032      +7.5%   0.8324

OVERALL WINNER
  Config:  n=2048, k=64
  N*:      top_n=400
  Scale*:  2.0
  KL:      2.1184
  SMACT Δ: +2.5%
  R²_norm: 0.8507
  Folder:  nf2048_k64
